# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata info
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We use the mlcroissant metadata API to inspect the various entities.

In [ ]:
# Explore dataset RecordSets
record_sets = list(dataset.metadata.record_sets)
print("RecordSet @id(s) and names:")
for rs in record_sets:
    # Each RecordSet entity
    print(f"  - @id: {rs['@id']}")
    print(f"    name: {rs.get('name', None)}")
    # List field IDs
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for fld in fields:
        print(f"      * @id: {fld['@id']}  | name: {fld.get('name','')}  | dataType: {fld.get('dataType','')}")
    print()

## 3. Data Extraction
Load data from record sets into DataFrames for analysis, referencing record set and field `@id`s as above.

In [ ]:
# Extract all records from all record sets
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print(f"Extracting data for RecordSets: {record_set_ids}\n")
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet: {record_set_id} – {df.shape[0]} records, columns: {df.columns.tolist()}")

# For exploration, pick the primary (central) record set by its @id
main_record_set_id = record_set_ids[0]
print(f"\nColumns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore and process the data: filter records, normalize numeric fields, and group by demographic features. 

_Referencing all columns/fields by their `@id` in the DataFrame columns._

In [ ]:
# Identify numeric and demographic field @ids from the Croissant schema (see overview above)
#
# Example assignments (replace with your dataset's actual field @id):
# Let's choose PHQ-9 total score as a numeric field, and gender or education for grouping

cols = dataframes[main_record_set_id].columns.tolist()
# Adapt based on output of previous cells (commonly IDs like 'phq9_score', 'GAD7_total', etc.)
numeric_field_id = None
for col in cols:
    if 'phq' in col.lower() and 'score' in col.lower():  # heuristic; adapt as per your dataset
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = cols[0]  # fallback
group_field_id = None
for col in cols:
    if 'gender' in col.lower() or 'sex' in col.lower():
        group_field_id = col
        break

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Basic filtering (e.g., PHQ-9 score > 10)
threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered rows where {numeric_field_id} > {threshold}: {filtered_df.shape[0]}")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic (e.g., gender), if available
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print("\nMean of", numeric_field_id, "by", group_field_id)
    print(grouped)
else:
    print(f"Warning: Group field '{group_field_id}' not found for grouping.")

## 5. Visualization
Visualize data distributions (e.g., histogram of PHQ-9 scores, distribution by gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
- We loaded and explored the FAIR² Mental Health Survey dataset using the `mlcroissant` library.
- All entities (record set, fields, columns) were referenced by their `@id`.
- Initial analysis on PHQ-9 scores and demographic features (such as gender) revealed how data can be filtered, normalized, and grouped for further research.

**Note:** For a deeper study, investigate additional fields and domain-specific relationships as described in the Croissant schema.